# 15.9 Debugging in Practice — Strategy, Bisection and Hard Bugs

**Prerequisites:** 15.1–15.8, 12 Concurrency, 14.11 Binary Search  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Debugging as a **method**, not a talent: reproduce → isolate → hypothesise → test
- 🔴 **Reproduce first** — everything else is guessing until you can
- **Bisection** — binary search (**14.11**) applied to commits, inputs and config
- **Delta debugging** — shrinking a failing input by hand, and what `hypothesis` automates
- Heisenbugs: concurrency, ordering, and the observer effect
- `tracemalloc` for memory that grows and never comes back
- A checklist for when you are truly stuck
- Interview questions

---

## Debugging is a method

The difference between someone who debugs quickly and someone who does not is almost never
knowledge of the tools. It is **refusing to guess**.

```
   ┌──────────────────────────────────────────────────────────────┐
   │  1. REPRODUCE   make it fail on demand                       │
   │        │        ── if you cannot, everything below is guessing│
   │        ▼                                                     │
   │  2. ISOLATE     shrink it: fewer lines, less data, one commit│
   │        │        ── bisection and delta debugging             │
   │        ▼                                                     │
   │  3. HYPOTHESISE one specific, falsifiable claim              │
   │        │        "the cap is applied before the jitter"       │
   │        ▼                                                     │
   │  4. TEST        the smallest experiment that could disprove it│
   │        │        ── print / pdb / a failing test              │
   │        ▼                                                     │
   │     confirmed? ── no ──> back to 3 with what you learned     │
   │        │ yes                                                 │
   │        ▼                                                     │
   │  5. FIX, then write the test that would have caught it       │
   └──────────────────────────────────────────────────────────────┘
```

🔴 **The step people skip is 3.** They jump from "something is wrong" to changing code, which
is not debugging — it is shuffling. The tell is changing two things at once: if it starts
working you do not know why, and you have learned nothing.

**One change per experiment. Write down what you expect before you run it.**

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py159_"))


def run_script(name, source, *args, flags=()):
    path = WORK / name
    path.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    done = subprocess.run([sys.executable, *flags, str(path), *args],
                          capture_output=True, text=True, encoding="utf-8",
                          errors="replace", timeout=180)
    return (done.stdout + done.stderr).rstrip()


print("scratch:", WORK)

## 1. Reproduce — and make it *deterministic*

"It fails sometimes" is not a reproduction. Before anything else, get to a command you can run
that fails **every time**.

The usual sources of "sometimes", and what to pin:

| Source | Pin it with |
|---|---|
| `random` | a fixed seed — `random.Random(42)` |
| the clock | an injected clock (**15.5**) |
| dict/set ordering across runs | `PYTHONHASHSEED=0`, or sort before comparing |
| test order | run the one test by node ID (**15.3**) |
| concurrency | reduce to one thread, or force the interleaving |
| external services | a fake (**15.5**) |

🔴 **`PYTHONHASHSEED` is the one people miss.** Python randomises string hashing per process, so
set iteration order changes between runs. Code that accidentally depends on it fails roughly
one run in *n* — the classic "works on my machine".

In [ ]:
HASH_ORDER = r"""
    tags = {"cache", "retry", "timeout", "backoff", "jitter"}
    # 🔴 A bug: this code assumes sets have a stable order.
    print("first tag:", next(iter(tags)))
"""

print("Three runs with the default (randomised) hash seed:")
for _ in range(3):
    print("  ", run_script("hashorder.py", HASH_ORDER))

print("\nThree runs with PYTHONHASHSEED pinned:")
import os
for _ in range(3):
    done = subprocess.run([sys.executable, str(WORK / "hashorder.py")],
                          capture_output=True, text=True, encoding="utf-8",
                          env={**os.environ, "PYTHONHASHSEED": "0"}, timeout=60)
    print("  ", done.stdout.strip())

The first three runs disagree; the pinned ones do not. If you have a test
that fails one time in five, **pin the seed before you do anything else** — then you have a
reproduction, and only then a bug you can chase.

> The same applies in reverse when hunting: once fixed, run **without** the pin, repeatedly, to
> prove the fix is real and not just a reordering.

## 2. Isolate by bisection

**14.11** taught binary search on a sorted list. Debugging is the same algorithm applied to
something else: you have a **boundary** between working and broken, and you halve the space
until you find it.

| You have | Bisect over | Tool |
|---|---|---|
| "it worked last month" | commits | **`git bisect`** |
| a 40 MB input that crashes | the input | delta debugging, below |
| a 300-line config | the settings | comment out half |
| 200 tests, one poisons another | the test set | `pytest` with halves of the suite |

### `git bisect`

```bash
git bisect start
git bisect bad                 # the current commit is broken
git bisect good v1.4.0         # this tag was fine
# git checks out the midpoint; you test and answer:
git bisect good   /   git bisect bad
...
git bisect reset
```

Over **1,000 commits that is 10 steps**, not 1,000. And it can be fully automatic — `git bisect
run pytest -x tests/test_retry.py` will find the offending commit while you make tea.

🔴 The requirement is a **reliable test**, which is why step 1 comes first. Bisection with a
flaky check will confidently blame an innocent commit.

In [ ]:
# Bisection, simulated: 1,000 "commits", one of which introduced the bug.
def make_history(size=1000, broke_at=637):
    """Each commit either passes the test or does not; the bug is introduced once."""
    return [i < broke_at for i in range(size)]


def bisect_history(history):
    """Find the first bad commit. Returns (index, number of tests run)."""
    low, high, tests = 0, len(history) - 1, 0
    while low < high:
        mid = (low + high) // 2
        tests += 1
        if history[mid]:          # still good here
            low = mid + 1
        else:                     # already bad here
            high = mid
    return low, tests


history = make_history()
culprit, tests_run = bisect_history(history)

print(f"commits in range      : {len(history):,}")
print(f"first bad commit      : {culprit}")
print(f"tests run to find it  : {tests_run}")
print(f"linear search would be: up to {len(history):,}")
print(f"speed-up              : {len(history) / tests_run:.0f}x")

# Verified against brute force, the 14.16 habit.
brute = next(i for i, ok in enumerate(history) if not ok)
print(f"\nagrees with brute force: {brute == culprit}")

Ten tests instead of a thousand. That is the entire argument for bisection,
and it is why `git bisect` is worth learning properly — it turns "when did this break?" from an
afternoon into ten minutes.

## Delta debugging — shrinking the input

Same idea, applied to **data**. You have a 5,000-row CSV that crashes the importer. Which row?

The manual algorithm: remove half. Still crashes? Keep going with that half. No longer crashes?
Put it back and remove the other half. Repeat until removing anything makes it pass.

That is exactly what `hypothesis` did automatically in **15.6** when it shrank a failing case to
`text='0', limit=0`. Here it is by hand, on an input the library did not generate.

In [ ]:
def process_batch(rows):
    """Import a batch of 'job_id=attempts' rows. Raises on a malformed row."""
    return {job: int(attempts) for job, attempts in (r.split("=") for r in rows)}


def still_fails(rows):
    try:
        process_batch(rows)
        return False
    except Exception:
        return True


def shrink(rows, fails):
    """Delta debugging: the smallest subset that still fails."""
    current, steps = list(rows), 0
    chunk = len(current) // 2
    while chunk >= 1:
        i = 0
        while i < len(current):
            candidate = current[:i] + current[i + chunk:]     # drop a chunk
            steps += 1
            if candidate and fails(candidate):
                current = candidate                            # it was not needed
            else:
                i += chunk
        if chunk == 1:
            break
        chunk //= 2
    return current, steps


import random

rng = random.Random(159)
batch = [f"build-{i}={rng.randint(0, 5)}" for i in range(400)]
batch.insert(287, "build-BAD")            # the one malformed row

print(f"failing input      : {len(batch)} rows")
minimal, steps = shrink(batch, still_fails)
print(f"shrunk to          : {len(minimal)} row(s) in {steps} checks")
print(f"the culprit        : {minimal}")
print(f"still fails        : {still_fails(minimal)}")
print(f"removing it passes : {not still_fails([r for r in batch if r != minimal[0]])}")

400 rows reduced to **the single row that matters**, automatically. A
one-row reproduction is something you can put straight into a test (**15.1**).

🔴 This is worth writing once and keeping. `shrink(input, predicate)` works on rows, on
characters, on config keys, on a list of installed packages — anything you can remove chunks
from. It is the highest-leverage twenty lines in this notebook.

## 3–4. Hypothesise and test

A hypothesis has to be **falsifiable** and **specific**. Compare:

| Not a hypothesis | A hypothesis |
|---|---|
| "the cache is broken" | "`cache_key()` returns the same key for two different regions" |
| "it's a race condition" | "two threads both read `count` before either writes it" |
| "something's wrong with the dates" | "`parse()` treats `03/04` as April 3rd in CI and March 4th locally" |

The right-hand ones each suggest **one experiment**, and either survive it or die. That is
progress; the left-hand ones cannot be wrong, so they teach nothing.

> **Write the expected result down before running the experiment.** If you do not, you will
> rationalise whatever you see. This is the whole reason the scientific method exists.

## 🔴 Heisenbugs — when looking changes the answer

Some bugs disappear when observed. They are the ones that cost days.

| Bug class | Why observing hides it |
|---|---|
| **Race conditions** (**12.2**) | `print` and `pdb` change the timing; the window closes |
| **Timing / timeouts** | the debugger's pauses exceed every timeout |
| **Uninitialised or freed memory** (C extensions) | a debug build zeroes what a release build left dirty |
| **Order dependence** (**15.4**) | running one test alone removes the interference |
| **Optimisation-dependent** | `python -O` strips your `assert` guards (**15.1**) |

The technique that works: **stop observing, start recording**. Log to memory, let it run at full
speed, and dump the log after it fails.

The next cell runs the *same* buggy counter four ways. Every version contains an identical
read-modify-write race. Only their **visibility** differs.

In [ ]:
RACE = r"""
    import sys
    import threading
    import time

    LOOPS = 60_000
    MODE = sys.argv[1]

    counter = 0


    def increment():
        global counter
        for _ in range(LOOPS):
            seen = counter                    # READ
            if MODE == "sleep0":
                time.sleep(0)                 # force a thread switch in the gap
            elif MODE == "busy":
                for _ in range(3):
                    pass                      # a little work, no switch
            elif MODE == "print":
                print("", end="")             # an "innocent" observation
            counter = seen + 1                # WRITE - the gap above is the bug


    threads = [threading.Thread(target=increment) for _ in range(4)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    expected = LOOPS * len(threads)
    print(f"  {MODE:7} expected {expected:,}  got {counter:,}  lost {expected - counter:,}")
"""

for mode, label in (("plain",  "as written, full speed"),
                    ("busy",   "a few no-op iterations in the gap"),
                    ("sleep0", "time.sleep(0) in the gap - forces a switch"),
                    ("print",  "a do-nothing print() in the gap")):
    print(f"{label}:")
    for _ in range(3):
        print(run_script("race.py", RACE, mode))
    print()

🔴 **Read that table of results carefully — it is the most important output in
this notebook.**

| Mode | Result | What it means |
|---|---|---|
| `plain` | **lost 0**, every run | the race is there and **never shows** |
| `busy` | 🔴 **usually 0 — but not always** | the true flaky bug: same code, same machine, different answer |
| `sleep0` | **loses most updates, every run** | `sleep(0)` yields, so the gap is *guaranteed* to be interrupted |
| `print` | loses a large, **varying** amount | observing changed the timing |

Four lessons, in order of importance:

1. 🔴 **Absence of a symptom is not absence of a bug.** The `plain` version is *wrong* — four
   threads doing read-modify-write with no lock — and it produced a perfect answer three times
   running. Ship it and it will fail on a different machine, a different CPython, or under load.
2. 🔴 **Look at `busy`.** Three identical runs, and at least one is likely to differ from the
   others. Nothing changed between them — not the code, not the machine, not the input. **This
   is what a flaky test actually is**, and why "I re-ran it and it passed" is worthless
   evidence (**15.6**).
3. **To reproduce a race, widen the window deliberately.** `time.sleep(0)` forces a thread
   switch exactly where the danger is, turning "fails once a week in production" into "fails
   every time" — which is step 1 of the loop, achieved on purpose.
4. **Observation changes the outcome.** The `print` version does not merely reveal the bug, it
   produces different numbers each run. A debugger does the same thing far more violently,
   which is why stepping through a race is futile.

> **These numbers will differ on your machine**, and `busy` in particular may come out all
> zeros or all losses. That variance *is* the subject — do not treat any single run of this
> cell as the result.

> **Why `plain` shows nothing on this machine.** CPython switches threads every few thousand
> bytecodes, and the read-modify-write here is only a handful — so the interpreter almost never
> switches inside the gap. The window is real but tiny. On a slower machine, with more threads,
> or on a free-threaded build, it is not.
>
> **12.2** has the fix (a `Lock`). What matters *here* is the method: this bug is found by
> reasoning about the gap and by deliberately widening it — never by stepping.

## Memory that grows: `tracemalloc`

A process that gets slower and fatter over days has no traceback. `tracemalloc` compares two
snapshots and tells you **which line** allocated what is still alive.

In [ ]:
print(run_script("leak.py", r"""
    import tracemalloc

    RESPONSE_CACHE = {}          # 🔴 never evicted - the bug


    def handle(request_id):
        RESPONSE_CACHE[request_id] = [0] * 2000
        return "ok"


    def transient(request_id):
        scratch = [0] * 2000     # allocated and released each call
        return len(scratch)


    tracemalloc.start()
    before = tracemalloc.take_snapshot()

    for i in range(400):
        handle(i)
        transient(i)

    after = tracemalloc.take_snapshot()

    print("growth since the first snapshot, by line:")
    for stat in after.compare_to(before, "lineno")[:3]:
        where = str(stat).split("\\")[-1]
        print("   ", where)

    current, peak = tracemalloc.get_traced_memory()
    print(f"\n  current {current/1024:,.0f} KiB | peak {peak/1024:,.0f} KiB")
"""))

The top line is `handle`'s cache insert, holding hundreds of KiB. `transient`
allocated just as much in total but **does not appear**, because its lists were freed — which is
exactly the distinction you need.

| Call | Use |
|---|---|
| `tracemalloc.start(n)` | begin, keeping `n` frames per allocation |
| `take_snapshot()` | a picture of what is currently allocated |
| `snapshot.compare_to(old, "lineno")` | **growth** between two points — the one you want |
| `get_traced_memory()` | current and peak totals |

🔴 Take the *first* snapshot after warm-up, not at process start — otherwise imports dominate
the list and hide your leak.

> Profiling for **speed** (`cProfile`, `timeit`) is a different job and belongs to
> **18 Tooling, Packaging and Environments**. This is memory *diagnosis*.

## When you are stuck

A checklist, roughly in order of how often it works:

1. **Read the error again.** Out loud. All of it, including the top half of a chained
   traceback (**15.7**).
2. **Check your assumptions with `p`, not with your memory.** Is that variable really what you
   think? Is that function really the one being imported (**15.5**, where-to-patch)?
3. **Are you even running the code you are editing?** A stale `.pyc`, the wrong virtualenv, an
   installed copy shadowing your source, an unsaved file. Add a deliberate syntax error and
   confirm it breaks.
4. **Reduce.** Delete code until it stops failing. The last thing you deleted is involved.
5. **Explain it out loud**, to a colleague or a rubber duck. Stating it forces the assumption
   you skipped into words.
6. **Check the boundaries**: empty, one, zero, negative, `None`, exactly-at-the-limit.
7. **`git diff` / `git log`.** What changed? It is almost always something that changed.
8. **Stop.** Sleep on it. This is not folklore — the fixation that blinds you to the obvious
   fades with a break.

🔴 And the one that saves the most time: **make sure the bug is where you think it is.** Hours
disappear into debugging correct code because the real fault was in the caller, the config, or
the data.

## Interview Questions

1. **Walk me through how you debug something.** *(reproduce → isolate → hypothesise → test →
   fix → test-that-catches-it. Naming a method is the answer.)*
2. **A test fails one run in ten. What do you do first?** *(make it deterministic — seed, clock,
   `PYTHONHASHSEED`, order. Never "re-run it".)*
3. **"It worked last release." How do you find what broke it?** *(`git bisect` — 10 steps over
   1,000 commits — with a reliable test.)*
4. **A 40 MB input crashes your parser. How do you find the bad record?** *(delta debugging;
   halve and re-test.)*
5. **You add a `print` and the bug goes away. What does that tell you?** *(timing dependence —
   a race, a timeout, or an ordering assumption. Record, don't observe.)*
6. **What is the difference between `__cause__` and `__context__`?** *(**15.7**: "direct cause"
   vs "during handling" — the second often means the handler itself is broken.)*
7. **When would you *not* reach for a debugger?** *(intermittent, production-only, concurrent,
   or hung — see the table in **15.8**.)*
8. **A service's memory grows over days. How do you diagnose it?** *(`tracemalloc` snapshot
   comparison, after warm-up.)*
9. **How would you get a stack trace out of a process that is hung, not crashed?**
   *(`faulthandler.dump_traceback_later`, or attach a debugger — **15.7**, **15.8**.)*
10. **You have fixed the bug. What is left to do?** *(write the failing test first, confirm it
    fails without the fix, then check whether the same mistake exists elsewhere.)*
11. **What is the most expensive debugging mistake you have made?** *(a real answer beats a
    tidy one — interviewers are testing whether you reflect.)*

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

---

## Common Mistakes & Pitfalls

1. 🔴 **Changing code before you can reproduce the failure.** Everything after that is guessing, and a fix you cannot verify is not a fix.
2. 🔴 **Changing two things at once.** If it starts working you have learned nothing and cannot undo half of it.
3. **Re-running a flaky test until it passes.** That is not a green build, it is a coin flip you keep tossing (**15.6**).
4. **Bisecting with an unreliable test.** `git bisect` will confidently name an innocent commit.
5. **Debugging code you are not actually running** — stale bytecode, wrong virtualenv, an installed copy shadowing your source. Prove it with a deliberate syntax error.
6. **Assuming the bug is where the traceback points.** It points at where it *surfaced*; the cause is often in the caller or the data.
7. **Using a debugger on a timing bug.** Stopping the world closes the window you are trying to see into.
8. **Taking the first `tracemalloc` snapshot at process start.** Imports dominate and hide the real growth.
9. **Fixing the symptom.** If you cannot explain *why* the fix works, you have moved the bug, not removed it.
10. **Not writing the test afterwards.** The same bug comes back, and next time you will not remember any of this.

## Best Practices

- Get to a deterministic reproduction before anything else — seed, clock, hash seed, single test by node ID.
- One change per experiment, with the expected result written down first.
- Reach for bisection whenever you have a working/broken boundary — commits, input rows, config keys, or the test set.
- Keep a `shrink(input, predicate)` helper; it pays for itself the first time you use it.
- State hypotheses so they can be *wrong* — vague ones cannot teach you anything.
- For timing bugs, record to memory and dump after the failure; do not observe live.
- Take `tracemalloc` snapshots after warm-up and compare, rather than reading absolute totals.
- When stuck, work the checklist rather than trying harder at the same thing.
- End every debugging session by writing the test that would have caught it (**15.1**).

## Practice Exercises

Try these before moving on.

1. Take a test of your own that uses `random` without a seed. Run it 20 times, then pin the seed and run it 20 more. Which failures were real?
2. 🔴 Set up a small repo with 30 commits where one introduces a bug, then find it with `git bisect run`. How many commits did it actually test?
3. Adapt this notebook's `shrink()` to reduce a *string* rather than a list of rows — find the single character that breaks a parser.
4. Take `15.6`'s `truncate` bug and shrink a failing input by hand. Compare your minimal case with the one `hypothesis` found.
5. 🔴 Run the race-condition cell again with `LOOPS` at 1,000 and at 1,000,000. Does `plain` ever lose an update? Then add a `Lock` (**12.2**) and confirm `sleep0` stops losing any. Which of the four modes would you have trusted as 'evidence it works'?
6. 🔴 Write a program that leaks memory through a module-level list, find it with `tracemalloc`, fix it, and prove the growth is gone.
7. Write down the last bug that took you more than an hour. Which step of the loop did you skip?
8. **Capstone:** deliberately break one function in a **14 Data Structure and Algorithm** notebook. Hand it to someone else with only the failing output. Time how long it takes them, and watch which step they skip.

---

## Version notes

| Version | Change |
|---|---|
| **3.14** | `python -m pdb -p PID` attaches to a running process — the missing piece for production hangs (**15.8**) |
| **3.13** | Coloured tracebacks; `tracemalloc` overhead reduced |
| **3.12** | `sys.monitoring` — much cheaper tracing, which is what makes modern coverage and profiling tools fast (**15.6**) |
| **3.11** | Fine-grained error locations (**15.7**) — the biggest single reduction in debugging time in years |
| **3.7+** | `breakpoint()` and `PYTHONBREAKPOINT` (**15.8**) |
| all | `PYTHONHASHSEED` has randomised string hashing since 3.3 — pin it to reproduce ordering bugs |

## 15 Testing and Debugging — the folder

| Notebook | Covers |
|---|---|
| **15.1** | why test; `assert`; `python -O`; AAA; a hand-rolled runner; the pyramid |
| **15.2** | `unittest` — `TestCase`, lifecycle, `subTest`, skips, discovery |
| **15.3** | `pytest` — assertion rewriting, `approx`, `raises`, `parametrize`, marks, the CLI |
| **15.4** | fixtures — scopes, `yield`, `conftest.py`, `tmp_path`, `monkeypatch` |
| **15.5** | test doubles — `Mock`, where to patch, `autospec`, fakes, injection |
| **15.6** | coverage, property-based testing, layout, flakiness, CI |
| **15.7** | tracebacks, chained exceptions, `excepthook`, logging, `faulthandler` |
| **15.8** | `pdb`, `breakpoint()`, post-mortem, `pytest --pdb` |
| **15.9** | this notebook — method, bisection, delta debugging, heisenbugs |

**The one-sentence version of the whole folder:** *tests tell you something is wrong, debugging
tells you why, and the last step of debugging is always writing the test.*

## Related

- **14.11 Searching and Binary Search** — bisection is that algorithm, applied to history
- **15.6** — flaky tests, and `hypothesis` shrinking, which delta debugging generalises
- **12.2 Threading** — the race condition demonstrated here, and its fix
- **15.5** — injecting a clock, so "sometimes" becomes "always"
- **18 Tooling, Packaging and Environments** — `cProfile` and profiling for speed